In [1]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from docx import Document

# Ensure reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Dataset directories
train_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Train'
val_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Val'

# Data augmentation
img_height, img_width = 224, 224
batch_size = 32

datagen_train = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, rotation_range=20, zoom_range=0.2, width_shift_range=0.2, height_shift_range=0.2
)
datagen_val = ImageDataGenerator(rescale=1./255)

train_data = datagen_train.flow_from_directory(
    train_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary'
)
val_data = datagen_val.flow_from_directory(
    val_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary', shuffle=False
)

# Function to save and display confusion matrix
def save_confusion_matrix(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix for {model_name}:\n{cm}")

    results_dir = 'D:\\PROJECT\\PROGRAM\\RESULTS\\'
    os.makedirs(results_dir, exist_ok=True)

    doc = Document()
    doc.add_heading(f'Confusion Matrix for {model_name}', level=1)
    for row in cm:
        doc.add_paragraph(' '.join(map(str, row)))
    doc.save(os.path.join(results_dir, f'{model_name}_confusion_matrix.docx'))

# Function to train and evaluate models
def train_and_evaluate(model, model_name):
    checkpoint_path = f'D:\\PROJECT\\PROGRAM\\MODELS\\{model_name}.keras'
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max')
    
    print(f"\nTraining {model_name}...\n")
    history = model.fit(train_data, epochs=20, validation_data=val_data, callbacks=[checkpoint])

    best_model = tf.keras.models.load_model(checkpoint_path)
    val_loss, val_accuracy = best_model.evaluate(val_data)
    print(f"{model_name} Validation Accuracy: {val_accuracy * 100:.2f}%")

    y_true = val_data.classes
    y_pred = (best_model.predict(val_data) > 0.5).astype('int32').flatten()

    save_confusion_matrix(model_name, y_true, y_pred)
    return val_accuracy

# EfficientNetB0 Model
efficientnet_base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
for layer in efficientnet_base.layers:
    layer.trainable = True  # Fine-tune all layers

efficientnet_model = Model(inputs=efficientnet_base.input, outputs=Dense(1, activation='sigmoid')(Flatten()(efficientnet_base.output)))
efficientnet_model.compile(optimizer=Adam(learning_rate=0.00001), loss='binary_crossentropy', metrics=['accuracy'])
efficientnet_accuracy = train_and_evaluate(efficientnet_model, 'EfficientNetB0_1')

# Print Results
print("\nEfficientNetB0 Accuracy:")
print(f"EfficientNetB0 Accuracy: {efficientnet_accuracy * 100:.2f}%")


Found 9900 images belonging to 2 classes.
Found 825 images belonging to 2 classes.

Training EfficientNetB0_1...



C:\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2388s 8s/step - accuracy: 0.5995 - loss: 0.7069 - val_accuracy: 0.4921 - val_loss: 0.7467
Epoch 2/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2358s 8s/step - accuracy: 0.7293 - loss: 0.5378 - val_accuracy: 0.7285 - val_loss: 0.5633
Epoch 3/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2297s 7s/step - accuracy: 0.7488 - loss: 0.5028 - val_accuracy: 0.7867 - val_loss: 0.4559
Epoch 4/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2240s 7s/step - accuracy: 0.7834 - loss: 0.4529 - val_accuracy: 0.8521 - val_loss: 0.3632
Epoch 5/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2234s 7s/step - accuracy: 0.8000 - loss: 0.4217 - val_accuracy: 0.7697 - val_loss: 0.4819
Epoch 6/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2218s 7s/step - accuracy: 0.8272 - loss: 0.3830 - val_accuracy: 0.8036 - val_loss: 0.4107
Epoch 7/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2242s 7s/step - accuracy: 0.8322 - loss: 0.3682 - val_accuracy: 0.8800 - val_loss: 0.2689
Epoch 8/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 2258s 7s/step - accuracy: 0.8474 - loss: 0.3425 - 